In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import to_timestamp
from pyspark.sql.functions import split, col, sum
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType

In [ ]:
# Create Spark session
#spark = SparkSession.builder.appName("ReadHDFSFile").getOrCreate()



#Create Spark session#
#spark = SparkSession.builder \
#    .config("spark.executor.memory", "10g") \
#    .config("spark.driver.memory", "10g") \
#    .config("spark.executor.extraJavaOptions", "-XX:+UseG1GC") \
#    .appName("ReadHDFSFile") \
#    .getOrCreate()


from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("ReadHDFSFile") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "1g") \
    .config("spark.executor.memoryOverhead", "512m") \
    .config("spark.sql.adaptive.enabled", "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .getOrCreate()


In [ ]:
# Read the file from HDFS (update path as needed)
df = spark.read.text("hdfs:///user/ubuntu/data/2019-08-22.txt")
# Show content of DataFrame
df.show()

In [ ]:
df.show(5, truncate=False)

In [ ]:

# Sample header based on the data
columns = ["transaction_id", "tx_datetime", "customer_id", "terminal_id", "tx_amount", "tx_time_seconds", "tx_time_days", "tx_fraud", "tx_fraud_scenario"]

# Filter out header row starting with #
df_data = df.filter(~col("value").startswith("#"))

# Split the "value" column by comma into array of strings
df_split = df_data.withColumn("split_values", split(col("value"), ","))

# Select each split part as a column
df_final = df_split.select(
    col("split_values").getItem(0).alias("transaction_id"),
    col("split_values").getItem(1).alias("tx_datetime"),
    col("split_values").getItem(2).alias("customer_id"),
    col("split_values").getItem(3).alias("terminal_id"),
    col("split_values").getItem(4).alias("tx_amount"),
    col("split_values").getItem(5).alias("tx_time_seconds"),
    col("split_values").getItem(6).alias("tx_time_days"),
    col("split_values").getItem(7).alias("tx_fraud"),
    col("split_values").getItem(8).alias("tx_fraud_scenario")
)

# Cast columns to correct data types as needed
df_typed = df_final.select(
    col("transaction_id").cast(IntegerType()),
    col("tx_datetime").cast(StringType()),
    col("customer_id").cast(IntegerType()),
    col("terminal_id").cast(IntegerType()),
    col("tx_amount").cast(DoubleType()),
    col("tx_time_seconds").cast(IntegerType()),
    col("tx_time_days").cast(IntegerType()),
    col("tx_fraud").cast(IntegerType()),
    col("tx_fraud_scenario").cast(IntegerType())
)

# Show structured data
df_typed.show(5)
df_typed.printSchema()


In [ ]:
total_rows = df_typed.count()
print(f"Total rows in the dataset: {total_rows}")


In [ ]:
# Count duplicate rows
duplicate_count = df_typed.groupBy(df_typed.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows: {duplicate_count}")


In [ ]:
# Show distinct counts per column (especially for categorical/ID columns)
for c in df_typed.columns:
    distinct_count = df_typed.select(col(c)).distinct().count()
    print(f"Distinct values in {c}: {distinct_count}")

In [ ]:
# Outliers detection tx_amount for numeric column
quantiles = df_typed.approxQuantile("tx_amount", [0.01, 0.99], 0.0)
lower_bound, upper_bound = quantiles
outliers = df_typed.filter((col("tx_amount") < lower_bound) | (col("tx_amount") > upper_bound))
outliers.show()

In [ ]:
from pyspark.sql.functions import col, count, when

df_typed.select([count(when(col(c).isNull(), c)).alias(c) for c in df_typed.columns]).show()

In [ ]:
# Check schema
expected_columns = ["transaction_id", "tx_datetime", "customer_id", "terminal_id", "tx_amount","tx_time_seconds", "tx_time_days", "tx_fraud", "tx_fraud_scenario"]
if not all(col in df_typed.columns for col in df_typed):
    print("Schema does not match expected columns")
